# Step 2 Diagnostics: SMH
How SMH's IDs link, how the PICANet file links to demographics, what the admission time means (local time against UTC), and whether the medication files are versions of one another. Counts only; nothing here changes any data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import diagnostics as D
import manifest as M
import picanet_times as PT
import smh_diagnostics as SD

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
reports_dir = open_picu_dir / "reports"
smh_dir = base / "SMH"

d = SD.load(smh_dir)
smh_m, smh_w = M.smh_manifest(smh_dir)

## 1. Demographics and the PICANet link
The demographics file repeats rows per encounter (exact duplicates, and some rows that differ, mostly in weight). `encounterId` is the stay key. PICANet starts in 2013, so earlier encounters are outside the cohort. `UniquePtID` is not the same string in the two files, so it cannot be used to link them.

In [ ]:
D.show(SD.demographics_duplicates(d))

In [ ]:
D.show(SD.picanet_link(d))

In [ ]:
D.show(SD.weight_checks(d))

## 2. Times
`inTime` is London local time and `utcChartTime` is UTC. If so, the first vitals row sits one hour before `inTime` in summer and at about zero in winter. The demographics-versus-PICANet time and LOS checks are in `picanet_times.py`.

In [ ]:
D.show(SD.time_offset(smh_dir, d))

In [ ]:
PT.check_times(smh_dir, "SMH", demo_adm="inTime", manifest=smh_m)

## 3. Medication files and drug coverage
`medications_v2` appears to be `medications` plus `medications_furosemide`. Then, for the eligible encounters, the share with at least one row in each drug table, by admission year, to see whether any table thins out.

In [ ]:
SD.medication_versions(smh_dir)

In [ ]:
SD.drug_coverage_by_year(smh_dir, smh_m)

## Exact duplicate rows in the SMH tables
Assembly removes rows that are identical in every column (see `SMH_findings.md`, question 5). Are they copies made by the extraction, or real repeated entries? For the intake and output table (`ai_picu_io`, where amounts are summed and a wrongly removed or kept copy would change totals) this counts how many rows are copies, how many times each row appears, how far apart the copies sit in the file, how many encounters are affected and how the rate changes by admission year. The number of copies should equal the exact duplicates removed in the assembly log (11,822 for this table). Counts only.

In [ ]:
smh_m = pd.read_csv(open_picu_dir / "derived" / "SMH_manifest.csv", dtype=str)
dup_io = SD.duplicate_pattern(smh_dir, smh_m, "ai_picu_io.csv")
D.show(dup_io)

## Exact duplicate rows in SMH vitals
Assembly removes rows that are identical in every column, and for vitals that is 3,441,560 of 8,449,349 rows in manifest stays (40.7%), the second largest share after the ventilation table. This is the table behind the hourly heart rate, blood pressure, temperature and saturation values, so it matters whether those rows are copies made by the extraction or real repeated readings. Same counts as for `ai_picu_io`: how many rows are copies, how many times each row appears, how far apart the copies sit in the file, how many encounters are affected and how the rate changes by admission year. The number of copies should equal the exact duplicates removed in the assembly log (3,441,560). Counts only. Closeness in the file does not by itself separate an extraction artefact from simultaneous real readings, because the time is one of the matching columns; the spread across encounters and years is the better evidence.

In [ ]:
dup_vitals = SD.duplicate_pattern(smh_dir, smh_m, "ai_picu_vitals.csv")
D.show(dup_vitals)

## 4. Save

In [ ]:
D.write_report({
    "Demographics": SD.demographics_duplicates(d), "PICANet link": SD.picanet_link(d),
    "Weight and time within an encounter": SD.weight_checks(d), "Time offset": SD.time_offset(smh_dir, d),
    "Medication versions": SD.medication_versions(smh_dir),
    "Drug coverage by year": SD.drug_coverage_by_year(smh_dir, smh_m),
    "Duplicate rows, ai_picu_io": dup_io,
    "Duplicate rows, ai_picu_vitals": dup_vitals,
}, reports_dir / f"SMH_diagnostics_{pd.Timestamp.today().date()}.md", title="SMH diagnostics")